In [1]:
import pandas as pd
import numpy as np
import os
import gc
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import warnings

# 警告を無視
warnings.filterwarnings('ignore')

# ==============================================================================
# 設定 (Config)
# ==============================================================================
# パスの自動判定
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'
else:
    INPUT_DIR = '.' 

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_simple_catboost.csv'

# ハイパーパラメータ
N_FOLDS = 5
RANDOM_SEED = 42

# 特徴量エンジニアリングの設定（ここが「なんとなくうまくいく」ポイント）
# 脳活動からレバー操作までには「タイムラグ」があり、
# かつ生データは「ノイズ」が多いため、以下で補正します。
LAG_LIST = [1, 3, 5, 10]    # 過去何フレーム前を見るか（反応遅れを捉える）
WINDOW_LIST = [5, 10]       # 過去何フレームの平均を取るか（ノイズを消してトレンドを見る）

# ==============================================================================
# 1. データ読み込み & 前処理
# ==============================================================================
print(">>> Loading Data...")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# ID系やターゲット以外の「脳活動データ列」を特定
ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
feature_cols = [c for c in train.columns if c not in ignore_cols]

# 欠損値埋め（CatBoostはNaN対応ですが、shiftで作られるNaNは埋めたほうが安全）
train = train.fillna(0)
test = test.fillna(0)

# ==============================================================================
# 2. 特徴量エンジニアリング (Heuristic Feature Extraction)
# ==============================================================================
print(">>> Feature Engineering (Lags, Rolling, Diff)...")

def engineer_features(df, features):
    """
    定石的な時系列特徴量を作成する関数
    """
    df_eng = df.copy()
    
    # 計算量削減のため、すべての列でやると重すぎる場合は
    # 重要そうな列だけに絞るのが一般的ですが、今回は全列やります（重い場合は列を絞ってください）
    target_features = features 
    
    for col in target_features:
        # sample_id ごとに計算しないと、別の実験データが混ざるので注意！
        grp = df_eng.groupby('sample_id')[col]
        
        # 1. Lag特徴量 (過去の値)
        # 「t秒前の脳の状態」が「今のレバー」に影響するため
        for lag in LAG_LIST:
            df_eng[f'{col}_lag{lag}'] = grp.shift(lag)
            
        # 2. Diff特徴量 (変化量)
        # 「値そのもの」より「急に上がったか」が重要な場合があるため
        df_eng[f'{col}_diff1'] = grp.diff(1)
        
        # 3. Rolling Mean (移動平均)
        # 生データはノイズが多いので、過去5回分の平均などで滑らかにする
        for window in WINDOW_LIST:
            # transformを使うと行数が変わらない
            df_eng[f'{col}_roll_mean{window}'] = grp.transform(lambda x: x.rolling(window).mean())
            
            # 余裕があれば標準偏差（ボラティリティ）も入れると効くことがあります
            # df_eng[f'{col}_roll_std{window}'] = grp.transform(lambda x: x.rolling(window).std())

    # shift()などで発生したNaNを0で埋める
    return df_eng.fillna(0)

# 特徴量生成実行（少し時間がかかります）
train = engineer_features(train, feature_cols)
test = engineer_features(test, feature_cols)

# 学習に使用する最終的な列リスト
use_features = [c for c in train.columns if c not in ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']]
print(f"Total Features: {len(use_features)}")

# メモリ解放
gc.collect()

# ==============================================================================
# 3. CatBoost 学習 (GroupKFold)
# ==============================================================================
print("\n>>> Starting CatBoost Training...")

# 時系列の「未来のリーク」を防ぐため、サンプル単位で分割するGroupKFoldを使用
kf = GroupKFold(n_splits=N_FOLDS)
groups = train['sample_id']

models = []
oof_preds = np.zeros(len(train))

for fold, (train_idx, val_idx) in enumerate(kf.split(train, groups=groups)):
    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
    
    X_train, y_train = train.iloc[train_idx][use_features], train.iloc[train_idx]['lever']
    X_val, y_val = train.iloc[val_idx][use_features], train.iloc[val_idx]['lever']
    
    train_pool = Pool(X_train, y_train)
    val_pool = Pool(X_val, y_val)
    
    # CatBoostパラメータ
    # GPUがあるなら task_type='GPU' にすると爆速になります
    model = CatBoostRegressor(
        iterations=5000,          # 多めに設定しEarly Stoppingに任せる
        learning_rate=0.05,       # 基本の学習率
        depth=6,                  # 深すぎず浅すぎず
        l2_leaf_reg=3,            # 正則化
        loss_function='RMSE',
        random_seed=RANDOM_SEED,
        early_stopping_rounds=50, # 改善しなくなったら止める
        verbose=500,
        task_type="GPU",          # GPU利用 (CPUの場合はここを削除または"CPU"に)
        devices='0'               # 1枚目を使用
    )
    
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    
    val_pred = model.predict(X_val)
    # レバー値は負にならないはずなので0でクリップ
    val_pred = np.maximum(0, val_pred)
    
    oof_preds[val_idx] = val_pred
    score = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f"Fold {fold+1} RMSE: {score:.4f}")
    
    models.append(model)
    
    del X_train, y_train, X_val, y_val, train_pool, val_pool
    gc.collect()

print(f"\n>>> Overall CV RMSE: {np.sqrt(mean_squared_error(train['lever'], oof_preds)):.4f}")

# ==============================================================================
# 4. 提出用ファイル作成
# ==============================================================================
print("\n>>> Predicting Test Data...")

test_preds = np.zeros(len(test))

# 全モデルの平均をとる（アンサンブル）
for model in models:
    preds = model.predict(test[use_features])
    test_preds += preds

test_preds /= len(models)
test_preds = np.maximum(0, test_preds)

submission = pd.DataFrame({
    'id': test['id'],
    'lever': test_preds
})

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved to {SUBMISSION_PATH}")

>>> Loading Data...
>>> Feature Engineering (Lags, Rolling, Diff)...
Total Features: 352

>>> Starting CatBoost Training...

--- Fold 1/10 ---
0:	learn: 1.3070956	test: 1.2824387	best: 1.2824387 (0)	total: 131ms	remaining: 10m 53s
500:	learn: 0.9919763	test: 1.0558585	best: 1.0558585 (500)	total: 4.79s	remaining: 43s
1000:	learn: 0.9331775	test: 1.0448282	best: 1.0448282 (1000)	total: 9.21s	remaining: 36.8s
bestTest = 1.04470576
bestIteration = 1004
Shrink model to first 1005 iterations.
Fold 1 RMSE: 1.0433

--- Fold 2/10 ---
0:	learn: 1.3043673	test: 1.3072127	best: 1.3072127 (0)	total: 20.4ms	remaining: 1m 42s
500:	learn: 0.9913834	test: 1.0801522	best: 1.0801522 (500)	total: 4.65s	remaining: 41.7s
1000:	learn: 0.9320164	test: 1.0675186	best: 1.0675186 (1000)	total: 9.08s	remaining: 36.3s
1500:	learn: 0.8896629	test: 1.0633474	best: 1.0633474 (1500)	total: 13.5s	remaining: 31.5s
bestTest = 1.061798295
bestIteration = 1647
Shrink model to first 1648 iterations.
Fold 2 RMSE: 1.0604

--

In [1]:
import pandas as pd
import numpy as np
import os
import gc
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import warnings

# 警告を無視
warnings.filterwarnings('ignore')

# ==============================================================================
# 設定 (Config)
# ==============================================================================
# パスの自動判定
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'
else:
    INPUT_DIR = '.' 

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_pid_catboost.csv'

# ハイパーパラメータ
N_FOLDS = 5
RANDOM_SEED = 42

# ==============================================================================
# 1. データ読み込み & 前処理
# ==============================================================================
print(">>> Loading Data...")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# ID系やターゲット以外の「脳活動データ列」を特定
ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
feature_cols = [c for c in train.columns if c not in ignore_cols]

# 欠損値埋め
train = train.fillna(0)
test = test.fillna(0)

# ==============================================================================
# 2. PID 特徴量エンジニアリング (PID Feature Extraction)
# ==============================================================================
print(">>> Feature Engineering (PID: Proportional, Integral, Derivative)...")

def engineer_pid_features(df, features):
    """
    PID制御の概念に基づいた特徴量を作成する関数
    P: 元の値
    I: 累積和 (Cumsum)
    D: 変化量 (Diff)
    """
    df_eng = df.copy()
    
    # グループ化（サンプルまたぎの計算を防ぐため）
    grouped = df_eng.groupby('sample_id')
    
    # ターゲットとなる特徴量ごとにPID成分を計算
    # ※列数が多い場合、ここでメモリを消費するので注意してください
    for col in features:
        # --- P項 (Proportional) ---
        # そのままの値（必要に応じてゲインを掛けることもありますが、今回は生値）
        # df_eng[f'{col}_P'] = df_eng[col] # 元の列があるので作成不要、明示したい場合のみ作成
        
        # --- I項 (Integral: 積分) ---
        # 過去からの蓄積。長期的なトレンドやバイアスを捉える
        df_eng[f'{col}_I'] = grouped[col].cumsum()
        
        # --- D項 (Derivative: 微分) ---
        # 変化の速度。急激なスパイクを捉える
        df_eng[f'{col}_D'] = grouped[col].diff().fillna(0)
        
        # --- D2項 (Second Derivative: 二階微分/加速度) ---
        # 変化の加速度。D項の変化量
        df_eng[f'{col}_D2'] = grouped[col].diff(2).fillna(0) # または .diff().diff()

    return df_eng

# 特徴量生成実行
train = engineer_pid_features(train, feature_cols)
test = engineer_pid_features(test, feature_cols)

# 学習に使用する最終的な列リストを更新
# 元の列(P相当) + I項 + D項 + D2項 が含まれる
use_features = [c for c in train.columns if c not in ignore_cols]
print(f"Total Features: {len(use_features)}")

# メモリ解放
gc.collect()

# ==============================================================================
# 3. CatBoost 学習 (GroupKFold)
# ==============================================================================
print("\n>>> Starting CatBoost Training...")

# 時系列の「未来のリーク」を防ぐため、サンプル単位で分割するGroupKFoldを使用
kf = GroupKFold(n_splits=N_FOLDS)
groups = train['sample_id']

models = []
oof_preds = np.zeros(len(train))

for fold, (train_idx, val_idx) in enumerate(kf.split(train, groups=groups)):
    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
    
    X_train, y_train = train.iloc[train_idx][use_features], train.iloc[train_idx]['lever']
    X_val, y_val = train.iloc[val_idx][use_features], train.iloc[val_idx]['lever']
    
    train_pool = Pool(X_train, y_train)
    val_pool = Pool(X_val, y_val)
    
    # CatBoostパラメータ
    # ★重要: 前回のエラー回避のため CPUモード にしています。
    # GPUが確実に使える環境なら task_type="GPU", devices='0' に戻してください。
    model = CatBoostRegressor(
        iterations=5000,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3,
        loss_function='RMSE',
        random_seed=RANDOM_SEED,
        early_stopping_rounds=50,
        verbose=500,
        task_type="CPU",         # 安全のためCPUに変更
        # devices='0'            # GPU用設定のためコメントアウト
    )
    
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    
    val_pred = model.predict(X_val)
    val_pred = np.maximum(0, val_pred)
    
    oof_preds[val_idx] = val_pred
    score = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f"Fold {fold+1} RMSE: {score:.4f}")
    
    models.append(model)
    
    del X_train, y_train, X_val, y_val, train_pool, val_pool
    gc.collect()

print(f"\n>>> Overall CV RMSE: {np.sqrt(mean_squared_error(train['lever'], oof_preds)):.4f}")

# ==============================================================================
# 4. 提出用ファイル作成
# ==============================================================================
print("\n>>> Predicting Test Data...")

test_preds = np.zeros(len(test))

# 全モデルの平均をとる（アンサンブル）
for model in models:
    preds = model.predict(test[use_features])
    test_preds += preds

test_preds /= len(models)
test_preds = np.maximum(0, test_preds)

submission = pd.DataFrame({
    'id': test['id'],
    'lever': test_preds
})

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved to {SUBMISSION_PATH}")

>>> Loading Data...
>>> Feature Engineering (PID: Proportional, Integral, Derivative)...
Total Features: 176

>>> Starting CatBoost Training...

--- Fold 1/5 ---
0:	learn: 1.3028408	test: 1.3137455	best: 1.3137455 (0)	total: 166ms	remaining: 13m 50s
500:	learn: 0.9743163	test: 1.0646102	best: 1.0646102 (500)	total: 38.5s	remaining: 5m 46s
1000:	learn: 0.8997489	test: 1.0514079	best: 1.0514038 (999)	total: 1m 15s	remaining: 5m 3s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.047450337
bestIteration = 1445

Shrink model to first 1446 iterations.
Fold 1 RMSE: 1.0460

--- Fold 2/5 ---
0:	learn: 1.3006350	test: 1.3226361	best: 1.3226361 (0)	total: 98.5ms	remaining: 8m 12s
500:	learn: 0.9810431	test: 1.0350764	best: 1.0350453 (498)	total: 38.1s	remaining: 5m 42s
1000:	learn: 0.9054141	test: 1.0209176	best: 1.0209176 (1000)	total: 1m 15s	remaining: 5m 1s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.01791591
bestIteration = 1314

Shrink model to fir

In [2]:
import pandas as pd
import numpy as np
import os
import gc
from scipy.stats import skew, kurtosis
from scipy.signal import welch
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 設定
# ==============================================================================
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'
else:
    INPUT_DIR = '.' 

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_hms_catboost.csv'

N_FOLDS = 5
RANDOM_SEED = 42

# ==============================================================================
# 1. データ読み込み
# ==============================================================================
print(">>> Loading Data...")
train = pd.read_csv(TRAIN_PATH).fillna(0)
test = pd.read_csv(TEST_PATH).fillna(0)

ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
feature_cols = [c for c in train.columns if c not in ignore_cols]

# ==============================================================================
# 2. HMS流 特徴量エンジニアリング
# ==============================================================================
print(">>> Feature Engineering (HMS Style: Time Stats, FFT, Hjorth)...")

def calculate_hms_features(df, features):
    """
    HMSコンペで多用されたEEG特徴量を抽出する関数
    """
    # 結果を格納する辞書のリスト（DataFrame生成用）
    # groupby().apply() は遅いので、numpyでループ処理するか、集計関数を一括適用します
    
    # 1. 基本統計量 (Time Domain)
    # groupbyオブジェクトを作成
    grp = df.groupby('sample_id')[features]
    
    # aggを使って一括計算 (高速)
    # mean, std, min, max, skew, kurt
    # ※skew, kurtosisはPandas標準にない場合があるため、applyを使うか自作が必要ですが、
    # ここでは計算速度優先で基本統計量に絞り、Hjorthで複雑性を補います。
    print("  - Calculating Basic Stats...")
    df_stats = grp.agg(['mean', 'std', 'max', 'min', 'var'])
    
    # カラム名をフラットにする (例: feature1_mean)
    df_stats.columns = [f'{col}_{stat}' for col, stat in df_stats.columns]
    
    # Peak-to-Peak (振幅)
    for col in features:
        df_stats[f'{col}_ptp'] = df_stats[f'{col}_max'] - df_stats[f'{col}_min']
    
    # 2. Hjorth Parameters & FFT (Frequency Domain)
    # これらはaggでやるのが難しいため、sample_idごとに処理
    # データ量が多いため、applyは重いです。主要な特徴量に絞って実装します。
    
    print("  - Calculating Spectral & Hjorth Features (This may take time)...")
    
    # sample_idごとの配列を取得
    # sample_idをインデックスにして、値をリスト化するのは重いので、
    # ここでは「差分(diff)」を使った簡易計算で代用します（ベクトル化による高速化）
    
    df_eng = df.copy()
    
    # 差分 (1階微分相当)
    df_diff = df_eng.groupby('sample_id')[features].diff().fillna(0)
    # 2階微分相当
    df_diff2 = df_eng.groupby('sample_id')[features].diff(2).fillna(0)
    
    # 元のデータセットに結合して集計
    for col in features:
        # --- Hjorth Activity ---
        # 分散 (var) は既に上で計算済み (df_stats[f'{col}_var'])
        
        # --- Hjorth Mobility ---
        # sqrt(var(diff(x)) / var(x))
        # まずdiffの分散を計算したいが、データフレーム全体でgroupbyしてaggするのは大変なので
        # 近似的に「diffの絶対値平均」などを特徴量として追加します
        
        # Line Length (線長): 波形の激しさ
        # sum(|x[t] - x[t-1]|)
        df_stats[f'{col}_line_length'] = df_diff[col].abs().groupby(df['sample_id']).mean()
        
        # --- Spectral Power (簡易版) ---
        # 本格的なFFTは重すぎるため、「ゼロ交差数 (Zero Crossings)」で周波数を近似
        # 値がプラスからマイナス（またはその逆）に変わった回数
        # (x[t] * x[t-1] < 0)
        
        # numpyで高速判定
        # signが変わった場所を検知
        # ここでは簡易的に「平均値との交差」を見ます
        # meanは sample_id ごとに違うので、centered data を作る必要があります
        
        # 代替案: 「高周波成分のパワー」として diffの二乗平均 を使う
        df_stats[f'{col}_high_freq_power'] = (df_diff[col] ** 2).groupby(df['sample_id']).mean()
        
    return df_stats

# 特徴量計算実行
# train_stats は sample_id をindexに持つDataFrameになります
train_stats = calculate_hms_features(train, feature_cols)
test_stats = calculate_hms_features(test, feature_cols)

# ターゲット変数(lever)を結合
# train_statsにはleverがないので、元のtrainから持ってくる必要があります
# sample_idごとのleverの代表値（平均など）ではなく、時系列予測なら
# 「そのsample_idのターゲット」が必要ですが、
# 今回のタスクは「時系列の各時刻の予測」ですか？それとも「サンプルごとの分類」ですか？
# -> これまでのコードを見ると「各時刻 (id)」ごとの予測です。

# ★重要★
# 上記の `calculate_hms_features` は「sample_id全体」を1つの特徴量に要約してしまいます。
# しかし、今回のタスクは「時系列の各ステップ」を予測する必要があります。
# したがって、要約した統計量（平均や分散）を、元の時系列データの各行に「ブロードキャスト（結合）」して、
# 「このサンプルは全体的にパワーが強い」といった文脈情報として使います。

print("  - Merging features back to time-series data...")
train_merged = train.merge(train_stats, on='sample_id', how='left')
test_merged = test.merge(test_stats, on='sample_id', how='left')

# PID特徴量（直近の変化）も重要なので、元の時系列データ(feature_cols)もそのまま使います。
# さらにHMS特徴量（大局的な統計情報）が加わる形です。

use_features = [c for c in train_merged.columns if c not in ignore_cols]
print(f"Total Features: {len(use_features)}")

# メモリ解放
del train_stats, test_stats, train, test
gc.collect()

# ==============================================================================
# 3. CatBoost 学習
# ==============================================================================
print("\n>>> Starting CatBoost Training (HMS Features)...")

kf = GroupKFold(n_splits=N_FOLDS)
groups = train_merged['sample_id']

models = []
oof_preds = np.zeros(len(train_merged))

for fold, (train_idx, val_idx) in enumerate(kf.split(train_merged, groups=groups)):
    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
    
    X_train, y_train = train_merged.iloc[train_idx][use_features], train_merged.iloc[train_idx]['lever']
    X_val, y_val = train_merged.iloc[val_idx][use_features], train_merged.iloc[val_idx]['lever']
    
    train_pool = Pool(X_train, y_train)
    val_pool = Pool(X_val, y_val)
    
    model = CatBoostRegressor(
        iterations=5000,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3,
        loss_function='RMSE',
        random_seed=RANDOM_SEED,
        early_stopping_rounds=50,
        verbose=500,
        task_type="CPU", # 安全のためCPU
    )
    
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    
    val_pred = model.predict(X_val)
    val_pred = np.maximum(0, val_pred)
    
    oof_preds[val_idx] = val_pred
    score = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f"Fold {fold+1} RMSE: {score:.4f}")
    
    models.append(model)
    gc.collect()

print(f"\n>>> Overall CV RMSE: {np.sqrt(mean_squared_error(train_merged['lever'], oof_preds)):.4f}")

# ==============================================================================
# 4. 提出
# ==============================================================================
print("\n>>> Predicting Test Data...")
test_preds = np.zeros(len(test_merged))

for model in models:
    preds = model.predict(test_merged[use_features])
    test_preds += preds

test_preds /= len(models)
test_preds = np.maximum(0, test_preds)

submission = pd.DataFrame({
    'id': test_merged['id'],
    'lever': test_preds
})

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved to {SUBMISSION_PATH}")

>>> Loading Data...
>>> Feature Engineering (HMS Style: Time Stats, FFT, Hjorth)...
  - Calculating Basic Stats...
  - Calculating Spectral & Hjorth Features (This may take time)...
  - Calculating Basic Stats...
  - Calculating Spectral & Hjorth Features (This may take time)...
  - Merging features back to time-series data...
Total Features: 396

>>> Starting CatBoost Training (HMS Features)...

--- Fold 1/5 ---
0:	learn: 1.3004149	test: 1.3114062	best: 1.3114062 (0)	total: 242ms	remaining: 20m 12s
500:	learn: 0.8441422	test: 1.0579988	best: 1.0579988 (500)	total: 1m 26s	remaining: 13m 1s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.049243485
bestIteration = 881

Shrink model to first 882 iterations.
Fold 1 RMSE: 1.0489

--- Fold 2/5 ---
0:	learn: 1.2979905	test: 1.3211130	best: 1.3211130 (0)	total: 235ms	remaining: 19m 32s
500:	learn: 0.8431698	test: 1.0454033	best: 1.0452920 (498)	total: 1m 27s	remaining: 13m 1s
1000:	learn: 0.7408836	test: 1.0341812	best: 1.0

In [1]:
import pandas as pd
import numpy as np
import os
import gc
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 設定
# ==============================================================================
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'
else:
    INPUT_DIR = '.' 

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_cmi_catboost.csv'

N_FOLDS = 5
RANDOM_SEED = 42

# CMIコンペで有効だったウィンドウサイズ（の脳波版）
# 短期的な反応(5)、中期的(20)、長期的(100)なトレンドを見る
WINDOW_SIZES = [5, 20, 100]

# ==============================================================================
# 1. データ読み込み
# ==============================================================================
print(">>> Loading Data...")
train = pd.read_csv(TRAIN_PATH).fillna(0)
test = pd.read_csv(TEST_PATH).fillna(0)

ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
feature_cols = [c for c in train.columns if c not in ignore_cols]

# ==============================================================================
# 2. CMI流 特徴量エンジニアリング (Rolling Stats & Envelope)
# ==============================================================================
print(">>> Feature Engineering (CMI Style: Rolling Mean/Std/Range, Abs, Log)...")

def engineer_cmi_features(df, features):
    """
    Sleep Statesコンペ由来の特徴量
    波形の「エネルギー」「ばらつき」のトレンドを捉える
    """
    df_eng = df.copy()
    
    # 1. Pre-processing: エネルギー(Abs) と 対数変換(Log)
    # 生データのプラスマイナスよりも「活動量」を見るため絶対値をとる
    for col in features:
        df_eng[f'{col}_abs'] = df_eng[col].abs()
        # スパイクの影響を抑えるログ変換 (Log1p: log(x+1))
        # 負の値がある場合はabsしてからlog
        df_eng[f'{col}_log_abs'] = np.log1p(df_eng[f'{col}_abs'])

    # 計算対象のカラムリストを更新（生データ + abs版）
    target_cols = features + [f'{c}_abs' for c in features]
    
    grouped = df_eng.groupby('sample_id')
    
    for col in target_cols:
        # 計算負荷軽減のため、主要な列のみに絞るなどの調整が実戦では必要ですが、
        # ここではループですべて回します。
        
        # 2. Rolling Statistics (移動統計量)
        for w in WINDOW_SIZES:
            # transformを使うことで行数を維持したまま計算
            # rolling().mean(): 平滑化（トレンド）
            df_eng[f'{col}_roll_mean_{w}'] = grouped[col].transform(lambda x: x.rolling(w, min_periods=1).mean())
            
            # rolling().std(): 変動の激しさ（Sleepコンペで最も重要だった特徴量の一つ）
            df_eng[f'{col}_roll_std_{w}'] = grouped[col].transform(lambda x: x.rolling(w, min_periods=1).std()).fillna(0)
            
            # rolling().max() - min(): 振幅の大きさ (Range)
            # 脳波が大きく振れている期間を特定
            roll_max = grouped[col].transform(lambda x: x.rolling(w, min_periods=1).max())
            roll_min = grouped[col].transform(lambda x: x.rolling(w, min_periods=1).min())
            df_eng[f'{col}_roll_range_{w}'] = roll_max - roll_min

    # 3. Time Features (時刻情報)
    # Sleepコンペでの「Hour」に相当。実験開始からの経過時間が「疲れ」や「慣れ」に影響する
    # time列が既にありますが、log変換したものも入れておきます
    df_eng['log_time'] = np.log1p(df_eng['time'])
    
    return df_eng

# 特徴量生成実行
# ※列数が爆発的に増えるので、メモリ注意です
train = engineer_cmi_features(train, feature_cols)
test = engineer_cmi_features(test, feature_cols)

# 学習に使用する列
use_features = [c for c in train.columns if c not in ignore_cols]
print(f"Total Features: {len(use_features)}")

# メモリ解放
gc.collect()

# ==============================================================================
# 3. CatBoost 学習
# ==============================================================================
print("\n>>> Starting CatBoost Training (CMI Features)...")

kf = GroupKFold(n_splits=N_FOLDS)
groups = train['sample_id']

models = []
oof_preds = np.zeros(len(train))

for fold, (train_idx, val_idx) in enumerate(kf.split(train, groups=groups)):
    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
    
    X_train, y_train = train.iloc[train_idx][use_features], train.iloc[train_idx]['lever']
    X_val, y_val = train.iloc[val_idx][use_features], train.iloc[val_idx]['lever']
    
    train_pool = Pool(X_train, y_train)
    val_pool = Pool(X_val, y_val)
    
    # ここでも安全のためCPUモード
    model = CatBoostRegressor(
        iterations=5000,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3,
        loss_function='RMSE',
        random_seed=RANDOM_SEED,
        early_stopping_rounds=50,
        verbose=500,
        task_type="CPU", 
    )
    
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    
    val_pred = model.predict(X_val)
    val_pred = np.maximum(0, val_pred)
    
    oof_preds[val_idx] = val_pred
    score = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f"Fold {fold+1} RMSE: {score:.4f}")
    
    models.append(model)
    gc.collect()

print(f"\n>>> Overall CV RMSE: {np.sqrt(mean_squared_error(train['lever'], oof_preds)):.4f}")

# ==============================================================================
# 4. 提出
# ==============================================================================
print("\n>>> Predicting Test Data...")
test_preds = np.zeros(len(test))

for model in models:
    preds = model.predict(test[use_features])
    test_preds += preds

test_preds /= len(models)
test_preds = np.maximum(0, test_preds)

submission = pd.DataFrame({
    'id': test['id'],
    'lever': test_preds
})

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved to {SUBMISSION_PATH}")

>>> Loading Data...
>>> Feature Engineering (CMI Style: Rolling Mean/Std/Range, Abs, Log)...
Total Features: 925

>>> Starting CatBoost Training (CMI Features)...

--- Fold 1/5 ---
0:	learn: 1.3029458	test: 1.3139838	best: 1.3139838 (0)	total: 598ms	remaining: 49m 49s
500:	learn: 0.9426890	test: 1.0903931	best: 1.0903931 (500)	total: 3m	remaining: 26m 59s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.087388871
bestIteration = 822

Shrink model to first 823 iterations.
Fold 1 RMSE: 1.0866

--- Fold 2/5 ---
500:	learn: 0.9479616	test: 1.0613787	best: 1.0613787 (500)	total: 3m 1s	remaining: 27m 6s
1000:	learn: 0.8446841	test: 1.0536720	best: 1.0536720 (1000)	total: 5m 59s	remaining: 23m 54s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.050372344
bestIteration = 1448

Shrink model to first 1449 iterations.
Fold 2 RMSE: 1.0496

--- Fold 3/5 ---
0:	learn: 1.3060255	test: 1.3020327	best: 1.3020327 (0)	total: 518ms	remaining: 43m 11s
500:	learn: 0.94